# Advanced Real Agent SDK Live Audit

This is an opt-in, paid integration harness. It uses real LLM calls, a real public MCP server, real project tools, a temporary workspace, real file edits, and real shell verification. It does not use a fake LLM or mocked tool results.

It validates the architecture expected from a modern coding-agent SDK: large catalogs stay deferred, tools activate only when relevant, malformed model calls are bounded and recoverable, local and MCP evidence can be combined, outputs are independently verified, and the same contract can be exercised across a model matrix.

## 1. Load this checkout first

Run Jupyter from this checkout with `.venv/bin/jupyter lab`, then execute cells from the top. The setup cell removes an already-imported site-packages copy and verifies that the current MCP constructor is loaded before any paid request.

In [1]:
import inspect
import sys
from pathlib import Path

EXPECTED_REPO = Path('/Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent')
repo = EXPECTED_REPO if EXPECTED_REPO.exists() else (
    Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
)
repo = repo.resolve()
loaded = sys.modules.get('shipit_agent')
loaded_path = Path(getattr(loaded, '__file__', '') or '/').resolve()
if loaded is not None and repo not in loaded_path.parents:
    stale_names = [
        name for name in sys.modules
        if name == 'shipit_agent' or name.startswith('shipit_agent.')
    ]
    for name in stale_names:
        sys.modules.pop(name, None)
    print(f'Removed stale SHIPIT import from {loaded_path}')
sys.path[:] = [str(repo)] + [item for item in sys.path if item != str(repo)]

import shipit_agent
from shipit_agent.mcp import RemoteMCPServer
from shipit_agent.models import AgentEvent
print('shipit-agent:', shipit_agent.__version__)
print('loaded from:', shipit_agent.__file__)
print('kernel:', sys.executable)
resolved_package = Path(shipit_agent.__file__).resolve()
if repo not in resolved_package.parents:
    raise RuntimeError(f'Wrong SHIPIT package loaded: {resolved_package}; expected {repo}')
if 'include_server_in_tool_names' not in inspect.signature(RemoteMCPServer).parameters:
    raise RuntimeError('Stale RemoteMCPServer loaded; restart the kernel and run cell 1 first')
preview_probe = repr(AgentEvent(type='usage_tick', message='probe', payload={'usage': {'prompt_tokens': 7}, 'tools': [{'name': 'probe'}]}))
if any(marker in preview_probe for marker in ('<int>', '<dict>', '<list>', '<bool>', '<float>')):
    raise RuntimeError('Stale SHIPIT classes are loaded. Restart the notebook kernel, then run all cells from the top.')
print('event preview:', preview_probe)


shipit-agent: 1.6.3
loaded from: /Users/rahulraj/Documents/MYWORK/ai_developer/others/shipit_agent/shipit_agent/__init__.py
kernel: /opt/homebrew/opt/python@3.11/bin/python3.11
event preview: AgentEvent(type='usage_tick', message='probe', payload={'usage': {'prompt_tokens': 7}, 'tools': [{'name': 'probe'}]}, timestamp=1786446305.098156)


## 2. Provider strategy

`LiteLLMChatLLM` is the common SHIPIT interface. Standard LiteLLM providers need only a model name and their normal environment credentials. Bedrock Mantle is an OpenAI-compatible Bedrock endpoint, but `bedrock-mantle/...` is not a stock LiteLLM provider identifier. The supplied `bedrock_mantle_provider.py` registers that transport with LiteLLM and normalizes Gemma-specific streamed tool-call spellings. It is a provider compatibility plug-in, not agent logic.

Three modes are supported below:

| Mode | Adapter | Use when |
| --- | --- | --- |
| `mantle-litellm` | `LiteLLMChatLLM` + registered provider | Current Gemma Mantle setup; strongest compatibility normalization |
| `mantle-native` | `BedrockGemmaChatLLM` | Direct OpenAI-compatible Mantle endpoint with `AWS_BEARER_TOKEN_BEDROCK` |
| `litellm` | `LiteLLMChatLLM` | Any normal LiteLLM model, including standard Bedrock, Anthropic, OpenAI, Gemini, or OpenRouter |

No credential is stored in the notebook. Configure environment variables before running it.

In [2]:
import importlib.util
import json
import os
import subprocess
import tempfile
from collections import Counter
from types import SimpleNamespace

from shipit_agent import Agent, DEEP_AGENT_PROMPT, InMemoryTraceStore
from shipit_agent.builtins import get_builtin_tool_map
from shipit_agent.deep import DeepAgent
from shipit_agent.llms import BedrockGemmaChatLLM, LiteLLMChatLLM
from shipit_agent.mcp import MCPStreamableHTTPTransport, RemoteMCPServer
from shipit_agent.prompts.default_agent_prompt import DEFAULT_AGENT_PROMPT

MODE = os.getenv('SHIPIT_LLM_MODE', 'mantle-litellm')
MODEL = os.getenv('SHIPIT_AUDIT_MODEL', 'bedrock-mantle/google.gemma-4-26b-a4b')
MAX_OUTPUT_TOKENS = int(os.getenv('SHIPIT_MAX_OUTPUT_TOKENS', '4096'))
AUDIT_TEMPERATURE = float(os.getenv('SHIPIT_AUDIT_TEMPERATURE', '0'))
LANGUAGE_CONTRACT = (
    '\n\n## Output language\nUse English only for public progress and final answers. '
    'Do not translate or repeat progress in another language.'
)
AUDIT_AGENT_PROMPT = DEFAULT_AGENT_PROMPT + LANGUAGE_CONTRACT
AUDIT_DEEP_PROMPT = DEEP_AGENT_PROMPT + LANGUAGE_CONTRACT
DEFAULT_PROVIDER = Path('/Users/rahulraj/Documents/MYWORK/AFTDRK/CACHE/DRK_CACHE_BACK/drk_cache/llm/bedrock_mantle_provider.py')
PROVIDER_PATH = Path(os.getenv('SHIPIT_MANTLE_PROVIDER', str(DEFAULT_PROVIDER)))

def register_mantle_provider():
    module_name = 'shipit_notebook_bedrock_mantle_provider'
    if module_name in sys.modules:
        return sys.modules[module_name]
    if not PROVIDER_PATH.exists():
        raise FileNotFoundError(f'Mantle provider not found: {PROVIDER_PATH}')
    spec = importlib.util.spec_from_file_location(module_name, PROVIDER_PATH)
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    module.ensure_registered()
    return module

def build_llm(model=MODEL, mode=MODE, **llm_kwargs):
    llm_kwargs.setdefault('temperature', AUDIT_TEMPERATURE)
    if mode == 'mantle-litellm' or model.startswith('bedrock-mantle/'):
        register_mantle_provider()
        return LiteLLMChatLLM(model=model, max_output_tokens=MAX_OUTPUT_TOKENS, **llm_kwargs)
    if mode == 'mantle-native':
        native_model = model.removeprefix('bedrock-mantle/')
        llm_kwargs.pop('prompt_cache_strategy', None)
        return BedrockGemmaChatLLM(model=native_model, max_tokens=MAX_OUTPUT_TOKENS, **llm_kwargs)
    return LiteLLMChatLLM(model=model, max_output_tokens=MAX_OUTPUT_TOKENS, **llm_kwargs)

print({'mode': MODE, 'model': MODEL, 'max_output_tokens': MAX_OUTPUT_TOKENS})


{'mode': 'mantle-litellm', 'model': 'bedrock-mantle/google.gemma-4-26b-a4b', 'max_output_tokens': 4096}


## 3. Real MCP and audit helpers

DeepWiki is a public Streamable HTTP MCP. Every call below creates a fresh server wrapper so tests do not share connection state.

In [3]:
DEEPWIKI_URL = 'https://mcp.deepwiki.com/mcp'

def deepwiki():
    return RemoteMCPServer(
        name='deepwiki',
        transport=MCPStreamableHTTPTransport(DEEPWIKI_URL, timeout=120),
        include_server_in_tool_names=True,
    )

def called_tools(result):
    return [
        str(event.payload.get('tool'))
        for event in result.events
        if event.type == 'tool_called' and event.payload.get('tool')
    ]

def stream_run(agent, prompt, *, label='agent'):
    """Print every event immediately and return a run-like audit object."""
    events = []
    print(f'\n=== LIVE STREAM: {label} ===', flush=True)
    for event in agent.stream(prompt):
        events.append(event)
        if event.type == 'text_delta':
            print(str(event.payload.get('chunk', '')), end='', flush=True)
            continue
        print(f'\n[{event.type}] {event.message}', flush=True)
        payload = dict(event.payload)
        if event.type == 'tool_output_delta':
            print(str(payload.pop('chunk', '')), flush=True)
            payload.pop('chunk_metadata', None)
        elif event.type == 'tool_completed':
            payload.pop('output', None)
            payload.pop('metadata', None)
        elif event.type in {'final_answer', 'run_completed'}:
            content = str(payload.pop('content', payload.pop('output', '')) or '')
            payload.pop('output', None)
            payload['content_chars'] = len(content)
        if payload:
            print(json.dumps(payload, indent=2, default=str), flush=True)
    completed = next(e.payload for e in reversed(events) if e.type == 'run_completed')
    runtime = getattr(agent, '_active_runtime', None)
    if runtime is None:
        runtime = getattr(getattr(agent, '_agent', None), '_active_runtime', None)
    metadata = dict(getattr(runtime, 'metadata', {}) or {})
    metadata.update(completed.get('tool_context') or {})
    print(f'\n=== STREAM COMPLETE: {label} ===', flush=True)
    return SimpleNamespace(
        output=str(completed.get('output', '') or ''),
        events=events,
        metadata=metadata,
    )

def audit(result, *, show_output=True):
    completed = next(e.payload for e in reversed(result.events) if e.type == 'run_completed')
    summary = {
        'tools': called_tools(result),
        'events': dict(Counter(e.type for e in result.events)),
        'usage': completed.get('usage'),
        'tool_context': completed.get('tool_context'),
        'prompt_cache': completed.get('prompt_cache'),
        'metadata': {
            key: result.metadata.get(key)
            for key in ('effective_code_mode', 'progressive_tool_context', 'hidden_tool_count', 'exposed_tool_count')
        },
    }
    print(json.dumps(summary, indent=2, default=str))
    if show_output:
        print('\nOUTPUT\n', result.output)
    return summary


## 4. Dormant-catalog test

A full built-in catalog plus MCP is available, but an exact-answer request must not execute unrelated tools. Auto mode should expose only a small stable gateway and keep the rest host-side.

In [4]:
workspace_handle = tempfile.TemporaryDirectory(prefix='shipit-live-audit-')
workspace = Path(workspace_handle.name)
(workspace / 'calculator.py').write_text('def divide(total, count):\n    return total * count\n')
(workspace / 'test_calculator.py').write_text(
    'from calculator import divide\n\ndef test_divide():\n    assert divide(12, 3) == 4\n'
)

80

The raw `print(event)` probe was removed because it ran the paid mixed MCP audit twice and printed one dataclass for every token. Section 7 uses `stream_run()`: text deltas render continuously, while decisions, tool groups, calls, outputs, usage, and completion events remain individually visible.

In [5]:
simple_agent = Agent.with_builtins(
    llm=build_llm(),
    prompt=AUDIT_AGENT_PROMPT,
    mcps=[deepwiki()],
    project_root='/tmp',
    auto_use_skills=False,
    trace_store=InMemoryTraceStore(),
    max_iterations=6,
)

for a in simple_agent.stream('Reply with exactly: hello'):
    print(a)

AgentEvent(type='run_started', message='Agent run started', payload={'prompt': 'Reply with exactly: hello'}, timestamp=1786446312.0257049)
AgentEvent(type='mcp_attached', message='MCP server attached: deepwiki', payload={'server': 'deepwiki'}, timestamp=1786446312.025713)
AgentEvent(type='step_started', message='LLM completion started', payload={'tool_count': 14, 'iteration': 1}, timestamp=1786446312.0277278)
AgentEvent(type='text_delta', message='', payload={'chunk': 'hello'}, timestamp=1786446313.6574159)
AgentEvent(type='usage_tick', message='Usage updated', payload={'usage': {'prompt_tokens': 3001, 'completion_tokens': 2, 'total_tokens': 3003, 'cache_read_input_tokens': 0, 'cache_creation_input_tokens': 0}, 'prompt_cache': {'provider': 'bedrock-mantle', 'model': 'bedrock-mantle/google.gemma-4-26b-a4b', 'supported': None, 'enabled': None, 'mode': 'provider_managed', 'reason': 'capability_not_declared', 'hit': None, 'usage_reported': False, 'read_tokens': 0, 'write_tokens': 0}, 'iter

In [ ]:
simple_agent = Agent.with_builtins(
    llm=build_llm(),
    prompt=AUDIT_AGENT_PROMPT,
    mcps=[deepwiki()],
    project_root='/tmp',
    auto_use_skills=False,
    trace_store=InMemoryTraceStore(),
    max_iterations=6,
)
simple = stream_run(simple_agent, 'Reply with exactly: hello', label='dormant catalog')
simple_audit = audit(simple)
actual = [name for name in called_tools(simple) if name not in {'tool_search', 'call_tool', 'todo', 'give_up'}]
assert actual == [], actual
assert 'hello' in simple.output.lower()
assert simple.metadata['progressive_tool_context'] is True
assert simple.metadata['effective_code_mode'] is False
assert simple.metadata['hidden_tool_count'] > simple.metadata['exposed_tool_count']


## 5. Deferred real MCP research

The same architecture must activate DeepWiki only when the question requires repository evidence. Programmatic `execute_code` remains absent unless `code_mode=True` is explicitly selected.

In [6]:
simple_agent =  Agent(
    llm=build_llm(),
    prompt=AUDIT_AGENT_PROMPT,
    mcps=[deepwiki()],
    project_root='/tmp',
    auto_use_skills=False,
    trace_store=InMemoryTraceStore(),
    tool_context_mode='auto',
    max_iterations=10,
)

for a in simple_agent.stream( 'Use DeepWiki to identify the openai/openai-python classes or methods '
    'that decide retry eligibility and backoff. Separate observed facts '
    'from inference and cite only tool evidence.'):
    print(a)

AgentEvent(type='run_started', message='Agent run started', payload={'prompt': 'Use DeepWiki to identify the openai/openai-python classes or methods that decide retry eligibility and backoff. Separate observed facts from inference and cite only tool evidence.'}, timestamp=1786446324.443518)
AgentEvent(type='mcp_attached', message='MCP server attached: deepwiki', payload={'server': 'deepwiki'}, timestamp=1786446324.443541)
AgentEvent(type='step_started', message='LLM completion started', payload={'tool_count': 2, 'iteration': 1}, timestamp=1786446324.4438488)
AgentEvent(type='text_delta', message='', payload={'chunk': 'I'}, timestamp=1786446325.1653278)
AgentEvent(type='text_delta', message='', payload={'chunk': ' do'}, timestamp=1786446325.166794)
AgentEvent(type='text_delta', message='', payload={'chunk': ' not'}, timestamp=1786446325.17261)
AgentEvent(type='text_delta', message='', payload={'chunk': ' have'}, timestamp=1786446325.173492)
AgentEvent(type='text_delta', message='', payl

In [ ]:
research_agent = Agent(
    llm=build_llm(),
    prompt=AUDIT_AGENT_PROMPT,
    mcps=[deepwiki()],
    project_root='/tmp',
    auto_use_skills=False,
    trace_store=InMemoryTraceStore(),
    tool_context_mode='auto',
    max_iterations=10,
)
research = stream_run(research_agent,
    'Use DeepWiki to identify the openai/openai-python classes or methods '
    'that decide retry eligibility and backoff. Separate observed facts '
    'from inference and cite only tool evidence.',
    label='deferred MCP research',
)
research_audit = audit(research)
assert any(name.startswith('deepwiki__') for name in called_tools(research))
assert 'execute_code' not in called_tools(research)
assert research_audit['usage']['total_tokens'] > 0


## 6. Real coding-agent repair

This creates a disposable repository, gives the agent only read/grep/edit/bash, requires a real patch and test run, then verifies the final workspace with an independent subprocess. The test passes only if the file on disk is correct.

In [16]:

tool_map = get_builtin_tool_map(llm=build_llm(), project_root=str(workspace))
coding_tools = [tool_map[name] for name in ('read_file', 'grep_files', 'edit_file', 'bash')]
coding_agent = Agent(
    llm=build_llm(),
    prompt=AUDIT_AGENT_PROMPT,
    tools=coding_tools,
    project_root=str(workspace),
    permission_mode='bypass',
    tool_context_mode='full',
    auto_use_skills=False,
    trace_store=InMemoryTraceStore(),
    max_iterations=14,
)

for a in coding_agent.stream(
    'Inspect calculator.py and its test. Fix the implementation bug with '
    'edit_file, then run pytest with bash. Do not change the test and do not '
    'finish until pytest passes.'
):
    print(a)
# coding = stream_run(coding_agent,
#     'Inspect calculator.py and its test. Fix the implementation bug with '
#     'edit_file, then run pytest with bash. Do not change the test and do not '
#     'finish until pytest passes.',
#     label='coding repair',
# )
# coding_audit = audit(coding)
# verification = subprocess.run(
#     [sys.executable, '-m', 'pytest', '-q'],
#     cwd=workspace, capture_output=True, text=True, check=False,
# )
# print(verification.stdout, verification.stderr)
# assert verification.returncode == 0
# assert 'return total / count' in (workspace / 'calculator.py').read_text()
# assert {'edit_file', 'bash'} <= set(called_tools(coding))


AgentEvent(type='run_started', message='Agent run started', payload={'prompt': 'Inspect calculator.py and its test. Fix the implementation bug with edit_file, then run pytest with bash. Do not change the test and do not finish until pytest passes.'}, timestamp=1786445825.197315)
AgentEvent(type='step_started', message='LLM completion started', payload={'tool_count': 4, 'iteration': 1}, timestamp=1786445825.197983)
AgentEvent(type='text_delta', message='', payload={'chunk': 'I'}, timestamp=1786445826.024485)
AgentEvent(type='text_delta', message='', payload={'chunk': ' will'}, timestamp=1786445826.024647)
AgentEvent(type='text_delta', message='', payload={'chunk': ' start'}, timestamp=1786445826.0247822)
AgentEvent(type='text_delta', message='', payload={'chunk': ' by'}, timestamp=1786445826.024894)
AgentEvent(type='text_delta', message='', payload={'chunk': ' listing'}, timestamp=1786445826.025017)
AgentEvent(type='text_delta', message='', payload={'chunk': ' the'}, timestamp=178644582

## 7. Mixed local + MCP evidence

This is the high-complexity routing test: local source must be inspected and remote repository behavior must be researched in one run. The assertion requires both tool families.

In [ ]:
(workspace / 'retry_policy.py').write_text(
    'class RetryPolicy:\n'
    '    max_attempts = 3\n'
    '    def should_retry(self, status):\n'
    '        return status in {429, 500, 502, 503}\n'
)
mixed_map = get_builtin_tool_map(llm=build_llm(), project_root=str(workspace))
mixed_tools = [mixed_map[name] for name in ('read_file', 'grep_files', 'bash')]
mixed_agent = Agent(
    llm=build_llm(), prompt=AUDIT_AGENT_PROMPT,
    tools=mixed_tools, mcps=[deepwiki()],
    project_root=str(workspace),
    tool_context_mode=os.getenv('SHIPIT_MIXED_TOOL_CONTEXT_MODE', 'full'),
    auto_use_skills=False, max_iterations=14,
    trace_store=InMemoryTraceStore(),
)
mixed = stream_run(mixed_agent,
    'Compare the local RetryPolicy with openai/openai-python retry behavior. '
    'You must inspect retry_policy.py and use DeepWiki. Separate observed '
    'facts from inference and propose two concrete tests.',
    label='mixed local and MCP evidence',
)
mixed_audit = audit(mixed)
mixed_called = called_tools(mixed)
assert any(name.startswith('deepwiki__') for name in mixed_called), mixed_called
assert set(mixed_called) & {'read_file', 'grep_files', 'bash'}, mixed_called
assert 'execute_code' not in mixed_called


## 8. DeepAgent live contract

DeepAgent wraps the same progressive runtime with planning/workspace capabilities. This cell is intentionally separate because it makes another paid, potentially longer call.

In [ ]:
RUN_DEEP = os.getenv('SHIPIT_RUN_DEEP_AUDIT', '0') == '1'
if RUN_DEEP:
    deep_agent = DeepAgent(
        llm=build_llm(), prompt=AUDIT_DEEP_PROMPT,
        extra_tools=mixed_tools, mcps=[deepwiki()],
        project_root=str(workspace), workspace_root=str(workspace / '.shipit'),
        permission_mode='bypass', auto_use_skills=False,
        tool_context_mode='auto', max_iterations=16,
        trace_store=InMemoryTraceStore(),
    )
    deep_result = stream_run(deep_agent,
        'Audit retry_policy.py against openai/openai-python using local and '
        'DeepWiki evidence. Produce a risk table and three executable test cases.',
        label='DeepAgent contract',
    )
    deep_audit = audit(deep_result)
    assert any(name.startswith('deepwiki__') for name in called_tools(deep_result))
    assert set(called_tools(deep_result)) & {'read_file', 'grep_files', 'bash'}
else:
    print('Skipped. Set SHIPIT_RUN_DEEP_AUDIT=1 to run the paid DeepAgent contract.')


## 9. Optional cross-model matrix

Set `SHIPIT_MODEL_MATRIX` to comma-separated LiteLLM model identifiers. Each model receives the same large-catalog no-tool contract. Mantle models automatically use the registered compatibility provider; normal LiteLLM identifiers use LiteLLM directly. This reveals model/provider capability differences instead of hiding them behind model-specific agent branches.

In [ ]:
matrix = [item.strip() for item in os.getenv('SHIPIT_MODEL_MATRIX', '').split(',') if item.strip()]
matrix_results = {}
for matrix_model in matrix:
    matrix_mode = 'mantle-litellm' if matrix_model.startswith('bedrock-mantle/') else 'litellm'
    candidate_agent = Agent.with_builtins(
        llm=build_llm(matrix_model, matrix_mode), mcps=[deepwiki()],
        prompt=AUDIT_AGENT_PROMPT,
        project_root='/tmp', auto_use_skills=False, max_iterations=6,
        trace_store=InMemoryTraceStore(),
    )
    candidate = stream_run(
        candidate_agent, 'Reply with exactly: hello', label=f'model matrix: {matrix_model}'
    )
    candidate_audit = audit(candidate, show_output=False)
    actual = [name for name in called_tools(candidate) if name not in {'tool_search', 'call_tool', 'todo', 'give_up'}]
    matrix_results[matrix_model] = {
        'passed': not actual and 'hello' in candidate.output.lower(),
        'actual_tools': actual,
        'usage': candidate_audit['usage'],
        'tool_context': candidate_audit['tool_context'],
    }
print(json.dumps(matrix_results, indent=2))
assert all(item['passed'] for item in matrix_results.values())


## 10. Prompt-cache capability matrix

This cell is offline and free. It verifies that SHIPIT chooses provider-safe behavior before any request is sent: explicit markers for translated providers, automatic caching where the provider owns it, provider-managed status for unknown routes, and a hard unsupported status for Bedrock Mantle Gemma.

In [ ]:
from shipit_agent.llms.base import prompt_cache_info

cache_policy_cases = {
    'native-mantle-gemma': BedrockGemmaChatLLM(
        model='google.gemma-4-31b', api_key='capability-probe'
    ),
    'litellm-openai': LiteLLMChatLLM(model='openai/gpt-5'),
    'litellm-anthropic': LiteLLMChatLLM(model='anthropic/claude-sonnet-4'),
    'litellm-bedrock-claude': LiteLLMChatLLM(
        model='bedrock/anthropic.claude-sonnet-4-5-20250929-v1:0'
    ),
    'litellm-gemini': LiteLLMChatLLM(model='gemini/gemini-2.5-flash'),
    'unknown-provider': LiteLLMChatLLM(model='custom/future-model'),
    'explicit-override': LiteLLMChatLLM(
        model='bedrock/future-cache-model', prompt_cache_strategy='explicit'
    ),
}
cache_policy_matrix = {
    name: prompt_cache_info(llm) for name, llm in cache_policy_cases.items()
}
print(json.dumps(cache_policy_matrix, indent=2))
assert cache_policy_matrix['litellm-openai']['mode'] == 'automatic'
assert cache_policy_matrix['litellm-bedrock-claude']['mode'] == 'explicit'
assert cache_policy_matrix['litellm-gemini']['mode'] == 'explicit'
assert cache_policy_matrix['unknown-provider']['mode'] == 'provider_managed'
assert cache_policy_matrix['native-mantle-gemma']['supported'] is False
assert cache_policy_matrix['explicit-override']['mode'] == 'explicit'


## 11. Real repeated-prefix cache audit

Set `SHIPIT_RUN_CACHE_AUDIT=1` to make two paid streamed calls with the same large stable system prefix. Set `SHIPIT_CACHE_MODEL` and `SHIPIT_CACHE_MODE` independently of the other notebook audits. For OpenAI-style models, `SHIPIT_CACHE_KEY` is forwarded as a routing hint. For a newly cache-capable LiteLLM route, set `SHIPIT_CACHE_STRATEGY=explicit`. Set `SHIPIT_REQUIRE_CACHE_HIT=1` in CI only when the selected provider/model is known to report cache usage reliably.

In [ ]:
RUN_CACHE_AUDIT = os.getenv('SHIPIT_RUN_CACHE_AUDIT', '0') == '1'
CACHE_MODEL = os.getenv('SHIPIT_CACHE_MODEL', MODEL)
CACHE_MODE = os.getenv('SHIPIT_CACHE_MODE', MODE)
CACHE_STRATEGY = os.getenv('SHIPIT_CACHE_STRATEGY', 'auto')
CACHE_KEY = os.getenv('SHIPIT_CACHE_KEY', 'shipit-live-audit-v1')
REQUIRE_CACHE_HIT = os.getenv('SHIPIT_REQUIRE_CACHE_HIT', '0') == '1'

def completed_payload(result):
    return next(e.payload for e in reversed(result.events) if e.type == 'run_completed')

def cache_ticks(result):
    return [
        e.payload.get('prompt_cache')
        for e in result.events
        if e.type == 'usage_tick' and e.payload.get('prompt_cache') is not None
    ]

if RUN_CACHE_AUDIT:
    cache_llm = build_llm(
        CACHE_MODEL, CACHE_MODE,
        prompt_cache_strategy=CACHE_STRATEGY,
        prompt_cache_key=CACHE_KEY,
    )
    stable_prefix = (
        'You are auditing a production agent SDK. Preserve this stable policy: '
        'separate observed facts from inference; do not call tools unless needed; '
        'answer with one short factual sentence. '
    ) * 160
    cache_agent = Agent(
        llm=cache_llm, prompt=stable_prefix, tools=[], auto_use_skills=False,
        max_iterations=2, trace_store=InMemoryTraceStore(),
    )
    cache_first = stream_run(
        cache_agent, 'State that this is cache probe one.', label='cache probe: first write'
    )
    cache_second = stream_run(
        cache_agent, 'State that this is cache probe two.', label='cache probe: repeated prefix'
    )
    first_done = completed_payload(cache_first)
    second_done = completed_payload(cache_second)
    cache_comparison = {
        'model': CACHE_MODEL,
        'strategy': CACHE_STRATEGY,
        'first': {
            'usage': first_done.get('usage'),
            'prompt_cache': first_done.get('prompt_cache'),
            'ticks': cache_ticks(cache_first),
        },
        'second': {
            'usage': second_done.get('usage'),
            'prompt_cache': second_done.get('prompt_cache'),
            'ticks': cache_ticks(cache_second),
        },
    }
    print(json.dumps(cache_comparison, indent=2, default=str))
    assert any(e.type == 'text_delta' for e in cache_first.events)
    assert any(e.type == 'text_delta' for e in cache_second.events)
    assert first_done.get('prompt_cache') is not None
    assert second_done.get('prompt_cache') is not None
    if REQUIRE_CACHE_HIT:
        assert second_done['prompt_cache']['hit'] is True, cache_comparison
else:
    print(
        'Skipped paid cache audit. Set SHIPIT_RUN_CACHE_AUDIT=1 and choose '
        'a cache-capable SHIPIT_CACHE_MODEL to run it.'
    )


## 12. What success means

A model is not considered coding-agent capable merely because it can answer chat. It must pass the observable contract: no irrelevant tool execution, deferred MCP activation, schema-valid tool calls, bounded completion output, correct local reads before edits, real workspace mutation, independent test success, and mixed local/remote evidence routing. Provider normalization belongs in the provider adapter; permissions, budgets, progressive discovery, retries, traces, and verification belong in the agent runtime.